<a href="https://colab.research.google.com/github/Chaman4211/flyrank-ml-internship/blob/main/work/notebooks/w02_ml_task_framing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Chaman4211/flyrank-ml-internship-starter/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [14]:
# Clone the repository to access the data files
!git clone https://github.com/Chaman4211/flyrank-ml-internship-starter.git

fatal: destination path 'flyrank-ml-internship-starter' already exists and is not an empty directory.


## 1. My lane as an ML task (type)

*Classification, clustering, ranking, or scoring — which one, and why?*

### Answer for 1. My lane as an ML task (type):

Based on the problem statement context of identifying declining content, this task most closely aligns with **Classification**. The goal is to predict whether a given content item will 'decline' or 'recover' (a binary outcome) within a future time window. This directly maps to the 'Will this one decline / recover?' question type from the `framing-ml-problems/SKILL.md` document.

In [15]:
# No specific code is typically required for defining the ML task type (Classification) itself at the framing stage.
# This cell serves as a placeholder for any code-based checks or queries related to the task definition,
# such as confirming the dataset's suitability, which is addressed by the data loading step.

## 2. Target or proxy

*What would you predict? Where does that label come from — observed outcome or a defined rule?*

### Answer for 2. Target or proxy:

The target for this classification task will be an **observed outcome** indicating content decline. A suitable target is `is_declining_label`. This label is derived from `trend_direction` (which itself comes from `trend_pct`) over a future period, representing an actual decrease in content performance that can be measured. It is an observed outcome rather than a rule-defined one, though we must be careful not to use the features it is derived from directly in the model. We are predicting *future* observed decline, not *current* decline based on a rule.

In [16]:
# Create the target variable 'is_declining_label' from the 'trend_direction' column.
# For binary classification, 'down' indicates decline (1), while 'stable' or 'up' indicate no decline (0).
import pandas as pd

df['is_declining_label'] = (df['trend_direction'] == 'down').astype(int)

print("Distribution of 'is_declining_label' (0=no decline, 1=decline):")
display(df['is_declining_label'].value_counts())

print(f"\nNaNs in 'is_declining_label': {df['is_declining_label'].isnull().sum()}")

Distribution of 'is_declining_label' (0=no decline, 1=decline):


,count
is_declining_label,
1,16262
0,13738



NaNs in 'is_declining_label': 0


## 3. Success metric

*One metric you can defend. What number means 'good'?*

### Answer for 3. Success metric:

Given that this is a classification task focused on identifying declining content, a suitable success metric is **ROC-AUC (Receiver Operating Characteristic - Area Under the Curve)**. This metric effectively evaluates the model's ability to distinguish between declining and non-declining content across all possible classification thresholds. It provides a single number representing the model's overall performance, which aligns with the guidance to "Name the metric before training" from the `framing-ml-problems/SKILL.md`.

In [17]:
# No specific code is required for defining the success metric (ROC-AUC) at this framing stage.
# The calculation of this metric will be implemented during model training and evaluation.
# This cell is a placeholder for potential checks, like confirming the target variable is binary (which is done in the target creation step).

## 4. The unit of analysis, as a real dataframe

*Load your lane's slice and show it: one row = one what?*

### Answer for 4. The unit of analysis:

The unit of analysis for this problem is a **single pseudonymized content item**. Each row in the `content_refresh_anonymized.csv` dataset represents a unique content item, with various features and historical performance metrics associated with it. The model will make a prediction for each individual content item.

In [18]:
import pandas as pd

# Load the starter dataset
df = pd.read_csv('flyrank-ml-internship-starter/data/raw/content_refresh_anonymized.csv')

# Display the first 5 rows to understand the structure
print("First 5 rows of the dataset:")
display(df.head())

# Display the shape of the dataframe
print(f"\nDataset shape: {df.shape[0]} rows, {df.shape[1]} columns")

First 5 rows of the dataset:


,content_id,client_id,search_volume,competition,competition_level,cpc,content_type,main_intent,word_count,char_count,...,char_count_tier,ctr,avg_position,engagement_rate,scroll_rate,ai_traffic_pct,impression_tier,position_tier,trend_direction,trend_pct
0,content_304f48230142,client_f369cb89fc,10.0,0.67,HIGH,2.05,keyword article,transactional,3221.0,20457.0,...,15000-25000,0.76,10.6,5.88,4.55,0.0,good,striking,down,-41.4
1,content_a1fb4e703a9e,client_4e07408562,90.0,0.01,LOW,0.05,keyword article,informational,2481.0,15562.0,...,15000-25000,0.05,20.3,0.00,10.00,0.0,good,page_3_5,down,-57.7
2,content_9aa793d4d895,client_7f2253d7e2,0.0,0.00,LOW,0.00,keyword article,informational,3515.0,23643.0,...,15000-25000,0.09,36.5,0.00,28.57,0.0,good,page_3_5,down,-60.9
3,content_331d6c4de07b,client_19581e27de,10.0,0.00,LOW,0.00,keyword article,commercial,NaN,NaN,...,NaN,0.49,6.2,1.28,3.45,0.0,good,page_1,stable,-13.8
4,content_d99b7a2d90ca,client_3fdba35f04,0.0,0.00,LOW,0.00,keyword article,informational,2803.0,17469.0,...,15000-25000,0.13,44.0,0.00,24.29,0.0,good,page_3_5,down,-34.7



Dataset shape: 30000 rows, 44 columns


## 5. Why ML beats a fixed rule here

*What makes the pattern too messy for an if-statement?*

### Answer for 5. Why ML beats a fixed rule here:

ML beats a fixed rule here because the pattern of content decline is **complex, non-linear, and influenced by numerous interacting factors** that would be impossible to capture with simple `if-then` statements. Content performance is dynamic, changing over time, and depends on a multitude of signals such as content type, historical engagement rates, keyword relevance, audience demographics, and external trends. A fixed rule would struggle to:

1.  **Account for interactions:** The interplay between different features (e.g., how `ctr` affects `scroll_rate` for a specific `content_type`) is too intricate for manual rules.
2.  **Adapt to changes:** Content trends and audience behavior evolve. A hard-coded rule would quickly become outdated.
3.  **Handle subtle signals:** Many weaker, correlated signals contribute to decline. ML can weigh these nuanced contributions, whereas a rule would need explicit thresholds for each.
4.  **Generalize:** A fixed rule might work for one content segment but fail catastrophically for others. ML models can learn more generalizable patterns across diverse content.

The 'messiness' of these relationships—many signals, tangled, and shifting over time—is precisely where ML earns its place.

In [19]:
# No specific code is required for explaining why ML beats a fixed rule.
# This section is conceptual, explaining the inherent complexity of the problem that warrants an ML solution.

## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.